# AI-Generated Video Detection Using CNN Features and LSTM-Based Temporal Modeling

| | |
|---|---|
| **Course** | Deep Learning — Final Project |
| **Student Name** | Safarov Fuadkhan |
| **Student ID** | 3096281 |

This notebook contains the full design, implementation, training, and evaluation of a custom deep learning
model for **AI-generated video detection**, as described in the accompanying project proposal.

### How this notebook maps to the assignment requirements

| Assignment requirement | Section(s) below |
|---|---|
| Problem Selection | §1 |
| Model Architecture (design + justification) | §4 |
| Data Collection/Preparation | §3 |
| Implementation (PyTorch, documented) | §3–§6 |
| Training and Evaluation (tuning + baseline comparison) | §5–§8 |
| Analysis (challenges, improvements, future work) | §9–§10 |
| Presentation deliverables | §11 |


## 1. Problem Selection

**Task.** Given a short video clip, classify it as **REAL** or **AI-GENERATED** (e.g., face-swapped or
otherwise synthetically manipulated).

**Why this problem.** The growing availability of face-manipulation and video-synthesis tools (DeepFaceLab,
First Order Motion Model, and other GAN-based systems) has made realistic fake video content easy to produce,
with real consequences for disinformation, fraud, and trust in visual media. As a course project, it is also
well suited to demonstrating two distinct deep learning ideas in one system: **spatial feature extraction**
from individual frames, and **temporal modeling** of how those features evolve across a sequence.

**Novelty / uniqueness of this application.** This project is not proposed as new research — combining a CNN
frame-level feature extractor with a recurrent temporal model for deepfake detection has prior precedent (e.g.
Güera & Delp, 2018, cited in §12). The unique contribution here is the **controlled, from-scratch, fully
documented implementation and comparison**: the same pretrained CNN backbone is used in two configurations —
(a) as an independent frame classifier with prediction averaging, and (b) as a feature extractor feeding an
LSTM — trained on the same data and evaluated under the same conditions, to directly answer whether temporal
modeling helps. This is the "unique application of existing models" framing: standard building blocks
(a pretrained CNN, an LSTM), combined and evaluated in a specific, self-contained pipeline designed for this
project.


## 2. Setup and Configuration

If running on Google Colab, select **Runtime → Change runtime type → GPU (T4)** before executing the cells below.


In [ ]:
# Kaggle notebooks already ship torch/torchvision/scikit-learn/matplotlib.
# opencv-python-headless decord and tqdm are the ones most likely missing.
!pip install -q opencv-python-headless decord tqdm


In [ ]:
import os
from pathlib import Path
import random
import glob
import numpy as np
import torch  # NOTE: must be imported before cv2/decord below -- on Windows, importing decord
              # first breaks torch's own DLL initialization (reproduced and confirmed during this
              # project: "OSError: DLL initialization routine failed" on torch's c10.dll). Keep
              # this import order if you add more cv2/decord-dependent code.
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import cv2
import decord

# Fix random seeds for reproducibility across runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


**Note (Kaggle adaptation):** the hyperparameters below are left identical to the local run so results stay comparable to the `2026-09-09_v1` baseline. With a full GPU and more data available here, `BATCH_SIZE`, `EPOCHS_BASELINE`, and `EPOCHS_LSTM` are worth raising once the pipeline is confirmed working end-to-end.

In [3]:
# ---- Project configuration ----
# All key hyperparameters live here so they are easy to find, change, and report on.

# DATA_DIR is a RELATIVE path, so this notebook works on any machine (Windows/Mac/Linux) and for
# anyone else who opens this same project folder -- no hardcoded username or absolute path needed.
# It resolves relative to wherever the notebook's kernel is running from (in VS Code, that's the
# folder containing this .ipynb file, by default). The two subfolders are created automatically
# below so you know exactly where to place the videos once you've downloaded them.
DATA_DIR = Path("data")          # expects DATA_DIR/real/*.mp4 and DATA_DIR/fake/*.mp4  (see Section 3)
(DATA_DIR / "real").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "fake").mkdir(parents=True, exist_ok=True)
print("Put real videos in: ", (DATA_DIR / "real").resolve())
print("Put fake videos in: ", (DATA_DIR / "fake").resolve())

NUM_FRAMES = 8                 # frames sampled per clip. Chosen as a balance between capturing enough
                                # temporal context for the LSTM and keeping training time/memory manageable
                                # on a single Colab GPU within the project's short timeframe.
IMG_SIZE = 224                 # standard input size for ImageNet-pretrained CNNs (ResNet18/EfficientNet-B0)
BATCH_SIZE = 4                  # kept small because each sample already contains NUM_FRAMES images
NUM_WORKERS = 2
EPOCHS_BASELINE = 8
EPOCHS_LSTM = 8
TUNE_EPOCHS = 2                 # short runs used only during the hyperparameter search in Section 7
DROPOUT_P = 0.3                 # regularization strength; deepfake datasets are comparatively small,
                                 # so moderate dropout reduces overfitting risk
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


Put real videos in:  C:\Users\User\Desktop\final project deeplearning\data\real
Put fake videos in:  C:\Users\User\Desktop\final project deeplearning\data\fake


## 3. Data Collection and Preparation *(Days 1–3)*

**Note:** this Kaggle variant was an unused fallback. The submitted results come from
`AI_Video_Detection_Project.ipynb`, which uses the DFDC sample set plus Google/Jigsaw's DFD dataset.

**Dataset:** the Kaggle [Deepfake Detection Challenge](https://www.kaggle.com/c/deepfake-detection-challenge)
(DFDC) training data, read directly from `/kaggle/input/deepfake-detection-challenge`. The cell below
extracts a chosen number of `dfdc_train_part_XX.zip` parts one at a time and sorts each video into
`data/real` or `data/fake` using that part's `metadata.json`. This requires having joined the competition,
which is no longer possible for new entrants.

### Expected folder structure

```
data/
├── real/
│   └── *.mp4
└── fake/
    └── *.mp4
```


In [ ]:
# ---- Assemble the dataset directly from the Kaggle-hosted DFDC competition data ----
# No download to your own machine happens here -- this reads straight from the
# read-only competition input that Kaggle attached to this notebook, and all
# extraction/processing happens on Kaggle's servers.
#
# The full DFDC training set ships as ~50 zip parts (dfdc_train_part_XX.zip),
# each with its own set of real+fake videos and its own metadata.json. This
# cell extracts a chosen number of parts one at a time (never all at once, to
# stay within the session's scratch disk), sorts each video into data/real or
# data/fake by its label, then deletes the raw extracted files before moving
# to the next part.

import glob
import zipfile
import json
import shutil
import os
from pathlib import Path

COMPETITION_INPUT_DIR = Path("/kaggle/input/deepfake-detection-challenge")

# STEP 1: see what's actually available before choosing how many parts to use.
# Run this cell once first, look at the printed listing, then adjust
# NUM_PARTS_TO_USE below and re-run.
available_files = sorted(os.listdir(COMPETITION_INPUT_DIR))
print(f"{len(available_files)} items in competition input directory:")
for f in available_files[:80]:
    print(" ", f)

# ---- How many DFDC training-data parts to pull in ----
# Each part is roughly 10GB zipped. Start small (e.g. 5) and raise this once
# you've confirmed the pipeline runs end-to-end; extracting all ~50 parts
# (~470GB total) will not fit in one Kaggle session's scratch disk.
NUM_PARTS_TO_USE = 5

part_zips = sorted(
    f for f in available_files
    if f.startswith("dfdc_train_part") and f.endswith(".zip")
)[:NUM_PARTS_TO_USE]
print("\nUsing parts:", part_zips)

if not part_zips:
    print(
        "\nNo files matched the 'dfdc_train_part*.zip' pattern above -- check the "
        "listing printed in STEP 1 and update the filter/pattern in this cell to "
        "match what's actually there."
    )

WORK_DIR = Path("/kaggle/working")
EXTRACT_TMP = WORK_DIR / "extract_tmp"
DATA_DIR = WORK_DIR / "data"
(DATA_DIR / "real").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "fake").mkdir(parents=True, exist_ok=True)

counts = {"REAL": 0, "FAKE": 0}

for zip_name in part_zips:
    print(f"\nExtracting {zip_name} ...")
    if EXTRACT_TMP.exists():
        shutil.rmtree(EXTRACT_TMP)
    EXTRACT_TMP.mkdir(parents=True)
    with zipfile.ZipFile(COMPETITION_INPUT_DIR / zip_name) as zf:
        zf.extractall(EXTRACT_TMP)

    # metadata.json's exact location can vary (directly inside, or nested one
    # folder deeper) -- search for it rather than assuming a fixed path.
    meta_paths = list(EXTRACT_TMP.rglob("metadata.json"))
    if not meta_paths:
        print(f"  WARNING: no metadata.json found inside {zip_name}, skipping this part")
        continue
    meta_path = meta_paths[0]
    part_root = meta_path.parent
    metadata = json.load(open(meta_path))

    for filename, info in metadata.items():
        src = part_root / filename
        if not src.exists():
            continue
        label = info["label"]
        dest_folder = "real" if label == "REAL" else "fake"
        dest = DATA_DIR / dest_folder / filename
        if dest.exists():
            continue  # filenames can repeat across parts; keep the first copy
        shutil.move(str(src), str(dest))
        counts[label] += 1

    shutil.rmtree(EXTRACT_TMP)  # free scratch disk before extracting the next part
    print(f"  Running totals -- REAL: {counts['REAL']}, FAKE: {counts['FAKE']}")

print("\nFinal counts across all parts used:", counts)


In [ ]:
# Sanity check: confirm the dataset landed where the rest of the notebook expects it
real_videos = sorted(glob.glob(os.path.join(DATA_DIR, "real", "*.mp4")))
fake_videos = sorted(glob.glob(os.path.join(DATA_DIR, "fake", "*.mp4")))
print(f"Real videos: {len(real_videos)}")
print(f"Fake videos: {len(fake_videos)}")

if len(real_videos) == 0 or len(fake_videos) == 0:
    print("\nWARNING: no videos found. Check NUM_PARTS_TO_USE and the file listing in the cell above.")


In [ ]:
# ---- Make sure results survive after the session ends ----
# Kaggle keeps whatever is in /kaggle/working when you click "Save Version";
# it then shows up under this notebook's "Output" tab for download, without
# ever having touched your own machine. Point CHECKPOINT_DIR-based saving at
# a working-dir path so it gets picked up automatically:
#   (already the case here since CHECKPOINT_DIR = "checkpoints" resolves under /kaggle/working)
print("Everything under /kaggle/working, including the checkpoints/ folder,")
print("will be downloadable from this notebook's Output tab after you Save Version.")


### Frame sampling

Rather than processing every frame (slow, and mostly redundant for short clips), we sample `NUM_FRAMES`
**evenly spaced** frames per video. This keeps every clip a fixed size for batching and gives the LSTM a
representative sketch of how the video evolves from start to end.


In [ ]:
# ---- Face detection setup ----
# The single biggest improvement found in this project came from cropping to the face before
# classifying, rather than feeding the whole video frame (background included) to the CNN.
# Uses cv2.FaceDetectorYN (YuNet) -- OpenCV's own modern, actively-maintained face detector.
# Note: on the machine this project was primarily developed on (OpenCV 5.0.0), the older
# Caffe-format face detectors and Haar cascades were both unavailable in that build; YuNet is
# also generally more accurate than either, so it's used here regardless of platform.
FACE_MODEL_DIR = Path("face_detector_model")
FACE_MODEL_PATH = FACE_MODEL_DIR / "face_detection_yunet_2023mar.onnx"
FACE_MODEL_URL = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"

if not FACE_MODEL_PATH.exists():
    import urllib.request
    FACE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Downloading face detector model to {FACE_MODEL_PATH} ...")
    urllib.request.urlretrieve(FACE_MODEL_URL, FACE_MODEL_PATH)

face_detector = cv2.FaceDetectorYN.create(str(FACE_MODEL_PATH), "", (320, 320), score_threshold=0.6)


def detect_face_box(frame_rgb, pad_frac=0.3):
    """Return a padded (x1, y1, x2, y2) box around the highest-confidence detected face,
    or None if no face is found (caller falls back to the full frame in that case)."""
    h, w = frame_rgb.shape[:2]
    frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
    face_detector.setInputSize((w, h))
    _, faces = face_detector.detect(frame_bgr)
    if faces is None or len(faces) == 0:
        return None
    best = max(faces, key=lambda f: f[-1])
    x, y, bw, bh = best[0], best[1], best[2], best[3]
    x1, y1, x2, y2 = int(x), int(y), int(x + bw), int(y + bh)
    pad_x, pad_y = int(bw * pad_frac), int(bh * pad_frac)
    x1, y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
    x2, y2 = min(w, x2 + pad_x), min(h, y2 + pad_y)
    if x2 <= x1 or y2 <= y1:
        return None
    return (x1, y1, x2, y2)


def sample_frame_indices(total_frames, num_frames):
    """Return `num_frames` evenly spaced frame indices across a video of length `total_frames`.

    If the video is shorter than `num_frames`, indices repeat (via linspace) so every clip still has a
    fixed length; this is handled gracefully rather than raising an error on short clips.
    """
    if total_frames <= num_frames:
        return np.linspace(0, max(total_frames - 1, 0), num_frames).astype(int)
    return np.linspace(0, total_frames - 1, num_frames).astype(int)


def load_video_frames(video_path, num_frames=NUM_FRAMES, img_size=IMG_SIZE):
    """Load `num_frames` evenly spaced, face-cropped RGB frames from a video, resized to
    (img_size, img_size). See the main project notebook for the full rationale -- decord for
    fast decode (RGB output, no color-conversion needed), face detection once per video reused
    across all sampled frames, graceful fallback to the full frame or a black frame if detection
    or decoding fails.
    """
    try:
        vr = decord.VideoReader(str(video_path), ctx=decord.cpu(0))
        total_frames = len(vr)
        indices = sample_frame_indices(total_frames, num_frames).tolist()
        frames_arr = vr.get_batch(indices).asnumpy()

        mid_frame = frames_arr[len(frames_arr) // 2]
        box = detect_face_box(mid_frame)

        frames = []
        for f in frames_arr:
            if box is not None:
                x1, y1, x2, y2 = box
                f = f[y1:y2, x1:x2]
            frames.append(cv2.resize(f, (img_size, img_size)))
    except Exception:
        frames = []

    while len(frames) < num_frames and len(frames) > 0:
        frames.append(frames[-1])
    if len(frames) == 0:
        frames = [np.zeros((img_size, img_size, 3), dtype=np.uint8)] * num_frames

    return np.stack(frames[:num_frames])


### Preprocessing and augmentation

- **Resize** every frame to 224×224 (required input size for the pretrained backbone).
- **Normalize** using standard ImageNet mean/std, since the CNN backbone was pretrained on ImageNet.
- **Augment** (train split only): random horizontal flip and mild brightness/contrast jitter. This is kept
  deliberately simple — heavier augmentation (compression simulation, rotation, etc.) is left as future work
  rather than a requirement (see §10).
- Face detection/cropping is **not** used by default; if the dataset already provides face-centered clips
  (as FF++ does), raw frames are used directly to keep the pipeline simple.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class VideoFrameDataset(Dataset):
    """Returns (clip, label, path) where clip has shape (T, C, H, W).

    label: 0 = real, 1 = AI-generated.
    """
    def __init__(self, video_paths, labels, transform, num_frames=NUM_FRAMES, img_size=IMG_SIZE):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
        self.num_frames = num_frames
        self.img_size = img_size

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        path = self.video_paths[idx]
        label = self.labels[idx]
        frames = load_video_frames(path, self.num_frames, self.img_size)   # (T, H, W, 3)

        transformed = [self.transform(frame) for frame in frames]          # list of (C, H, W)
        clip = torch.stack(transformed, dim=0)                             # (T, C, H, W)
        return clip, torch.tensor(label, dtype=torch.float32), path


### Train / validation / test split

The split is performed **at the video level**, not the frame level, so frames from the same source video
never appear in more than one split. This avoids identity/source leakage that would otherwise make the
reported metrics artificially optimistic.


In [ ]:
def split_videos(video_paths, labels, val_split=VAL_SPLIT, test_split=TEST_SPLIT, seed=SEED):
    """Split video paths/labels into train/val/test, stratified by class.

    Splitting the combined list as one shuffled block would let the real/fake ratio in each split
    vary with the random seed -- with only 77 real videos in the DFDC sample, an unlucky shuffle could
    leave val or test with very few (or zero) real examples. Splitting each class separately at the
    same val_split/test_split fractions keeps the class ratio consistent across train/val/test
    regardless of seed or dataset size.
    """
    class_indices = {}
    for i, label in enumerate(labels):
        class_indices.setdefault(label, []).append(i)

    train_idx, val_idx, test_idx = [], [], []
    for label, indices in class_indices.items():
        shuffled = indices[:]
        random.Random(seed).shuffle(shuffled)

        n = len(shuffled)
        n_test = int(n * test_split)
        n_val = int(n * val_split)

        test_idx.extend(shuffled[:n_test])
        val_idx.extend(shuffled[n_test:n_test + n_val])
        train_idx.extend(shuffled[n_test + n_val:])

    def subset(indices):
        return [video_paths[i] for i in indices], [labels[i] for i in indices]

    return subset(train_idx), subset(val_idx), subset(test_idx)


all_paths = real_videos + fake_videos
all_labels = [0] * len(real_videos) + [1] * len(fake_videos)   # 0 = real, 1 = fake

(train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels) = split_videos(all_paths, all_labels)

print(f"Train videos: {len(train_paths)} | Val videos: {len(val_paths)} | Test videos: {len(test_paths)}")

train_ds = VideoFrameDataset(train_paths, train_labels, transform=train_transform)
val_ds = VideoFrameDataset(val_paths, val_labels, transform=eval_transform)
test_ds = VideoFrameDataset(test_paths, test_labels, transform=eval_transform)

# --- Class imbalance handling (train split only; val/test keep the natural distribution so
# metrics stay honest) ---
# 1) WeightedRandomSampler: with BATCH_SIZE=4 and real videos only ~20% of train, plain random
#    shuffling would give a large fraction of batches zero real examples at all (no gradient signal
#    for that class that step). Sampling real/fake with equal probability per draw fixes that.
n_real_train, n_fake_train = train_labels.count(0), train_labels.count(1)
class_count = {0: n_real_train, 1: n_fake_train}
train_sample_weights = [1.0 / class_count[label] for label in train_labels]
train_sampler = WeightedRandomSampler(train_sample_weights, num_samples=len(train_labels), replacement=True)

# 2) pos_weight for BCEWithLogitsLoss: label 1 = fake here, so the standard imbalance formula
#    (pos_weight = negative_count / positive_count = real / fake) comes out < 1, which *down*-weights
#    the abundant fake-class loss terms relative to the scarce real-class ones.
POS_WEIGHT = n_real_train / n_fake_train
print(f"Train class counts -> real: {n_real_train}, fake: {n_fake_train} | POS_WEIGHT: {POS_WEIGHT:.4f}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


## 4. Model Architecture *(Days 4–10)*

Two models are implemented so the value of temporal modeling can be measured directly, rather than assumed.

```
VIDEO
  |
Sample 8 frames, resize + normalize
  |
+-----------------------------+-----------------------------+
|        Baseline path        |        Main model path       |
|   CNN -> per-frame logit    |   CNN -> per-frame feature   |
|   -> average over frames    |   -> LSTM -> final hidden    |
|   -> REAL / AI              |   -> Dropout -> FC -> REAL/AI|
+-----------------------------+-----------------------------+
```

### Design justification

- **CNN backbone — ResNet18, ImageNet-pretrained.** Training a CNN from scratch is impractical here: deepfake
  datasets are small relative to what large CNNs need, and the project timeframe (2–3 weeks) does not allow
  for it. Transfer learning from ImageNet gives the model useful low- and mid-level visual features (edges,
  textures) from the start, which is standard practice for this data/time regime. ResNet18 is chosen over
  deeper variants (ResNet50, EfficientNet-B0) as a lighter, faster-to-train option well suited to a single
  Colab GPU and a short schedule; EfficientNet-B0 is noted as a drop-in alternative if more capacity is needed.
- **Temporal module — a single-layer LSTM** (main model only). An LSTM is chosen over a Transformer for this
  core implementation because it is simpler to implement and debug, trains faster on short 8-frame sequences,
  and needs less data to fit well — all relevant given the project's short timeframe. A Transformer encoder
  is noted as a future extension (§10), not a requirement for the working system.
- **Activation functions.** The LSTM's internal gates use their standard sigmoid/tanh nonlinearities (built
  into `nn.LSTM`). No extra activation is added between the LSTM output and the classifier, since the final
  layer produces a single raw logit consumed by `BCEWithLogitsLoss` (which applies sigmoid internally for a
  numerically stable loss computation).
- **Dropout (p=0.3)** is added before each model's final linear layer. This is the one regularization detail
  not explicitly spelled out in the proposal's architecture description, added here specifically to reduce
  overfitting risk given the comparatively small dataset — a standard, minimal addition that does not change
  the overall data flow described in the proposal.
- **Hidden size (LSTM) = 256** by default, chosen as a reasonable middle ground between model capacity and
  overfitting risk for a small dataset; this is one of the values explored in the hyperparameter search in §7.


In [ ]:
class CNNBaselineClassifier(nn.Module):
    """Frame-level CNN baseline.

    Video -> sampled frames -> pretrained CNN -> per-frame logit -> average over frames -> REAL / AI

    Each frame is classified independently; there is no temporal reasoning at all. This establishes how much
    a purely frame-level model can achieve, and is the reference point the CNN+LSTM model must beat.
    """
    def __init__(self, pretrained=True, dropout_p=DROPOUT_P):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        backbone.fc = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(backbone.fc.in_features, 1),   # single logit: real (0) vs fake (1)
        )
        self.backbone = backbone

    def forward(self, x):
        # x: (B, T, C, H, W) -> flatten frames into the batch dimension for the CNN
        B, Tn, C, H, W = x.shape
        x = x.view(B * Tn, C, H, W)
        frame_logits = self.backbone(x).view(B, Tn)   # (B, T)
        clip_logit = frame_logits.mean(dim=1)          # average over frames -> (B,)
        return clip_logit


In [ ]:
class CNNLSTMClassifier(nn.Module):
    """Custom CNN + LSTM temporal classifier.

    Video -> sampled frames -> pretrained CNN (feature extractor) -> sequence of feature vectors
          -> LSTM -> final hidden state -> Dropout -> fully connected layer -> REAL / AI

    The CNN backbone (final classification layer removed) produces a 512-d feature vector per frame.
    The sequence of feature vectors, in temporal order, is passed through an LSTM, which can learn patterns
    across frames (e.g. inconsistencies that only become visible over several frames rather than in any
    single one). The LSTM's final hidden state is passed through a fully connected layer for the prediction.
    """
    def __init__(self, hidden_size=256, num_lstm_layers=1, dropout_p=DROPOUT_P, pretrained=True):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        self.feature_dim = backbone.fc.in_features     # 512 for ResNet18
        backbone.fc = nn.Identity()                      # use as a feature extractor, not a classifier
        self.cnn = backbone

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_lstm_layers,
            batch_first=True,
        )
        self.dropout = nn.Dropout(p=dropout_p)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, Tn, C, H, W = x.shape
        x = x.view(B * Tn, C, H, W)
        features = self.cnn(x)                                    # (B*T, feature_dim)
        features = features.view(B, Tn, self.feature_dim)          # (B, T, feature_dim)

        lstm_out, (h_n, c_n) = self.lstm(features)
        final_hidden = self.dropout(h_n[-1])                        # (B, hidden_size)
        logit = self.classifier(final_hidden).squeeze(1)             # (B,)
        return logit


## 5. Training Utilities

Both models are trained the same way, following the standard supervised deep learning workflow:
`dataset → dataloader → model → loss → backpropagation → optimizer → validation → testing`.

- **Loss:** `BCEWithLogitsLoss` (binary cross-entropy on raw logits — numerically stable, and pairs naturally
  with a single-logit output).
- **Optimizer:** Adam, a standard, robust default that needs little manual tuning.
- **Regularization:** dropout (§4) plus early stopping on validation loss, implemented below.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Run one epoch. Trains if `optimizer` is given, otherwise evaluates without updating weights."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for clips, labels, _ in tqdm(loader, leave=False):
            clips, labels = clips.to(DEVICE), labels.to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            logits = model(clips)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * clips.size(0)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_preds.extend((probs > 0.5).astype(int).tolist())
            all_labels.extend(labels.detach().cpu().numpy().astype(int).tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds) if len(all_labels) > 0 else 0.0
    return avg_loss, acc


def train_model(model, train_loader, val_loader, epochs, lr, checkpoint_path, patience=3, verbose=True,
                 pos_weight=None, min_epochs_before_checkpoint=2):
    """Full training loop with Adam, BCEWithLogitsLoss, and early stopping on validation loss.

    `pos_weight` (optional) is passed straight through to BCEWithLogitsLoss to counter class
    imbalance in the training data (see the POS_WEIGHT computed in Section 3, from train_labels
    counts) -- validation loss is intentionally left unweighted so early stopping/checkpointing
    still tracks genuine held-out performance, not a re-weighted proxy for it.

    `min_epochs_before_checkpoint`: epochs before this one are not eligible to be saved as the
    "best" checkpoint (the model still trains through them normally). With a validation split this
    small (~50-60 samples for this project), validation loss on the very first epoch(s) is driven
    mostly by random-init noise rather than genuine learning, and can spuriously look lower than
    later epochs where the model has actually learned something useful. Without this guard, that
    early noisy dip gets permanently saved as "best" and everything the model learns afterward is
    discarded -- which is what was happening to the LSTM run in analysis/2026-09-09_v1 (best
    checkpoint ended up being epoch 1, val_acc 0.42, before training had really started).

    Saves the best-performing checkpoint (by validation loss, among eligible epochs) and reloads it
    before returning, so the returned model is always the best eligible one seen during training,
    not necessarily the last epoch.
    """
    model.to(DEVICE)
    if pos_weight is not None:
        train_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))
    else:
        train_criterion = nn.BCEWithLogitsLoss()
    val_criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, train_criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, val_criterion, optimizer=None)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if verbose:
            print(f"Epoch {epoch}/{epochs} | "
                  f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        eligible = epoch >= min_epochs_before_checkpoint
        if eligible and val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), checkpoint_path)
        elif eligible:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch} (no val improvement for {patience} epochs).")
                break

    # Fallback: if training ended before any eligible epoch ever ran (epochs < min_epochs_before_checkpoint),
    # nothing would have been saved yet -- save the last epoch's weights so loading below doesn't fail.
    if not os.path.exists(checkpoint_path):
        torch.save(model.state_dict(), checkpoint_path)

    model.load_state_dict(torch.load(checkpoint_path))
    return model, history


## 6. Train the Baseline Model *(Days 4–6)*


In [ ]:
baseline_model = CNNBaselineClassifier(pretrained=True)
baseline_model, baseline_history = train_model(
    baseline_model, train_loader, val_loader,
    epochs=EPOCHS_BASELINE, lr=1e-4,
    checkpoint_path=os.path.join(CHECKPOINT_DIR, "baseline_best.pt"),
    pos_weight=POS_WEIGHT,
)


## 7. Hyperparameter Tuning — CNN + LSTM *(Days 7–10)*

To satisfy the "tuning hyperparameters to optimize performance" requirement without an extensive/expensive
search, a small, manually chosen grid is evaluated: two LSTM hidden sizes and two learning rates (3 combinations
total), each trained for a short `TUNE_EPOCHS` run. The configuration with the lowest validation loss is then
trained for the full `EPOCHS_LSTM` epochs in §8. This is a deliberately light-weight search — a broader,
systematic search (e.g. with Optuna) is noted as future work in §10 rather than attempted here.


In [ ]:
tuning_configs = [
    {"hidden_size": 128, "lr": 1e-4},
    {"hidden_size": 256, "lr": 1e-4},
    {"hidden_size": 256, "lr": 5e-5},
]

tuning_results = []
for cfg in tuning_configs:
    print(f"Tuning trial: hidden_size={cfg['hidden_size']} lr={cfg['lr']:.0e}")
    trial_model = CNNLSTMClassifier(hidden_size=cfg["hidden_size"])
    trial_ckpt = os.path.join(CHECKPOINT_DIR, f"tune_h{cfg['hidden_size']}_lr{cfg['lr']}.pt")
    _, trial_history = train_model(
        trial_model, train_loader, val_loader,
        epochs=TUNE_EPOCHS, lr=cfg["lr"], checkpoint_path=trial_ckpt, verbose=False,
        pos_weight=POS_WEIGHT,
    )
    final_val_loss = trial_history["val_loss"][-1]
    tuning_results.append({**cfg, "val_loss": final_val_loss})
    print(f"  -> val_loss={final_val_loss:.4f}")

best_cfg = min(tuning_results, key=lambda r: r["val_loss"])
print(f"\nBest configuration found: {best_cfg}")


## 8. Train the Final CNN + LSTM Model *(Days 7–10, continued)*

Using the best configuration from §7, train the main model for the full number of epochs with early stopping.


In [ ]:
lstm_model = CNNLSTMClassifier(hidden_size=best_cfg["hidden_size"])
lstm_model, lstm_history = train_model(
    lstm_model, train_loader, val_loader,
    epochs=EPOCHS_LSTM, lr=best_cfg["lr"],
    checkpoint_path=os.path.join(CHECKPOINT_DIR, "lstm_best.pt"),
    pos_weight=POS_WEIGHT,
)


## 9. Evaluation and Comparison *(Days 11–13)*

Both models are evaluated on the same held-out **test set**, which neither model has seen during training or
hyperparameter tuning. This section produces the central result of the project: **does the CNN+LSTM model
outperform the frame-averaging baseline?**


In [ ]:
def get_predictions(model, loader):
    """Run a trained model over a loader and collect probabilities, hard predictions, labels, and paths."""
    model.eval()
    all_probs, all_preds, all_labels, all_paths = [], [], [], []
    with torch.no_grad():
        for clips, labels, paths in tqdm(loader, leave=False):
            clips = clips.to(DEVICE)
            logits = model(clips)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.numpy().astype(int).tolist())
            all_paths.extend(paths)
    return np.array(all_probs), np.array(all_preds), np.array(all_labels), all_paths


def compute_metrics(labels, preds, probs, name):
    metrics = {
        "model": name,
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }
    try:
        metrics["roc_auc"] = roc_auc_score(labels, probs)
    except ValueError:
        metrics["roc_auc"] = float("nan")   # only one class present in this split
    return metrics


In [ ]:
baseline_probs, baseline_preds, test_labels, test_paths = get_predictions(baseline_model, test_loader)
lstm_probs, lstm_preds, test_labels_lstm, _ = get_predictions(lstm_model, test_loader)

baseline_metrics = compute_metrics(test_labels, baseline_preds, baseline_probs, "CNN Baseline")
lstm_metrics = compute_metrics(test_labels_lstm, lstm_preds, lstm_probs, "CNN + LSTM")

print(f"{'Metric':<12}{'CNN Baseline':<15}{'CNN + LSTM':<15}")
for key in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    print(f"{key:<12}{baseline_metrics[key]:<15.4f}{lstm_metrics[key]:<15.4f}")


In [ ]:
print("=== CNN Baseline - classification report ===")
print(classification_report(test_labels, baseline_preds, target_names=["real", "fake"]))

print("=== CNN + LSTM - classification report ===")
print(classification_report(test_labels_lstm, lstm_preds, target_names=["real", "fake"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, preds, name in zip(axes, [baseline_preds, lstm_preds], ["CNN Baseline", "CNN + LSTM"]):
    cm = confusion_matrix(test_labels, preds)
    ax.imshow(cm, cmap="Blues")
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["real", "fake"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["real", "fake"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(baseline_history["train_loss"], label="Baseline train")
axes[0].plot(baseline_history["val_loss"], label="Baseline val")
axes[0].plot(lstm_history["train_loss"], label="CNN+LSTM train")
axes[0].plot(lstm_history["val_loss"], label="CNN+LSTM val")
axes[0].set_title("Loss curves"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(baseline_history["train_acc"], label="Baseline train")
axes[1].plot(baseline_history["val_acc"], label="Baseline val")
axes[1].plot(lstm_history["train_acc"], label="CNN+LSTM train")
axes[1].plot(lstm_history["val_acc"], label="CNN+LSTM val")
axes[1].set_title("Accuracy curves"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


## 10. Error Analysis *(Days 14–17)*

Inspecting the CNN+LSTM model's mistakes helps identify where it still struggles — e.g. specific manipulation
types, heavily compressed clips, or short/ambiguous sequences — which directly feeds the discussion in §11.


In [ ]:
misclassified_idx = np.where(lstm_preds != test_labels_lstm)[0]
print(f"CNN+LSTM misclassified {len(misclassified_idx)} / {len(test_labels_lstm)} test clips.")

for i in misclassified_idx[:10]:
    true_label = "fake" if test_labels_lstm[i] == 1 else "real"
    pred_label = "fake" if lstm_preds[i] == 1 else "real"
    print(f"- {os.path.basename(test_paths[i])} | true={true_label} pred={pred_label} "
          f"prob_fake={lstm_probs[i]:.3f}")


In [ ]:
# Visualize the sampled frames of one misclassified example
if len(misclassified_idx) > 0:
    example_path = test_paths[misclassified_idx[0]]
    frames = load_video_frames(example_path, NUM_FRAMES, IMG_SIZE)

    fig, axes = plt.subplots(1, NUM_FRAMES, figsize=(2 * NUM_FRAMES, 2))
    for ax, frame in zip(axes, frames):
        ax.imshow(frame)
        ax.axis("off")
    fig.suptitle(f"Sampled frames - {os.path.basename(example_path)}")
    plt.tight_layout()
    plt.show()


**Discussion prompts to fill in after running on the real dataset:**
- Do misclassifications cluster around a particular manipulation method, or are they spread evenly?
- Do misclassified real videos share visible qualities (e.g. unusual lighting, low resolution) with the fake
  class, or vice versa?
- Does the CNN+LSTM model make different mistakes than the baseline, or largely the same ones?


## 11. Analysis and Discussion *(Days 14–17, continued)*

### 11.1 Results summary
*(Fill in after running: which model performed better, on which metrics specifically, and by how much. State
plainly if the LSTM did **not** outperform the baseline — that is still a valid, reportable finding, and
§11.2/§11.3 below can help explain why.)*

### 11.2 Challenges faced and how they were addressed

- **Limited compute/time.** Training was constrained to a short 2–3 week window on typically a single Colab
  GPU. Addressed by using a lightweight pretrained backbone (ResNet18), a short frame count (`NUM_FRAMES=8`),
  small batch sizes, and a deliberately light hyperparameter search (§7) rather than an exhaustive one.
- **Small dataset relative to model capacity.** Deepfake datasets are small compared to what CNNs are usually
  trained on. Addressed via transfer learning (ImageNet-pretrained backbone), dropout regularization, and
  early stopping on validation loss to avoid overfitting.
- **Risk of identity/source leakage between splits.** Naively splitting by frame (rather than by video) can
  let the model "memorize" a specific identity's appearance rather than learning general artifacts. Addressed
  by splitting at the video level (§3), so no source video appears in more than one split.
- **Class imbalance / metric choice.** Accuracy alone can be misleading if real/fake counts differ. Addressed
  by reporting precision, recall, F1, and ROC-AUC alongside accuracy, and by inspecting the confusion matrix
  directly rather than relying on a single number.
- *(Add any dataset-access, environment, or debugging challenges actually encountered while running this
  notebook on the real dataset.)*

### 11.3 Limitations

- Trained and evaluated on a single primary dataset (FaceForensics++); generalization to unseen manipulation
  methods or newer diffusion-based generators was not tested.
- No face detection/cropping was used in the core pipeline, so some frames may include background regions
  unrelated to the manipulation itself.
- The hyperparameter search (§7) covered only two hyperparameters over three combinations — enough to
  demonstrate the tuning workflow, but not an exhaustive search.
- Clip length is fixed at 8 frames; longer or variable-length clips were not explored.

### 11.4 Future work

- Replace the LSTM with a Transformer encoder and compare temporal modeling strategies.
- Evaluate on a second dataset (e.g. Celeb-DF) to check cross-dataset generalization.
- Test robustness to video re-compression at different quality levels.
- Add face detection/cropping (e.g. MTCNN) as an explicit preprocessing step.
- Run a broader hyperparameter search (learning-rate schedules, number of sampled frames, LSTM depth).


## 12. Deliverables and Presentation Checklist *(Days 18–21)*

**Before submitting:**
- [ ] Re-run the full notebook top-to-bottom on the real dataset to confirm it executes cleanly end-to-end.
- [ ] Fill in the placeholders in §11.1 (results summary) and §11.2 (any additional challenges encountered).
- [ ] Set the notebook's sharing permission so it is accessible to everyone who needs to grade it.
- [ ] Record a 10–15 minute video presentation covering: the problem (§1), the architecture and why it was
      designed this way (§4), the training/tuning approach (§7–§8), the results comparison (§9), and the
      conclusions and future work (§11.4).
- [ ] Upload the video to YouTube and collect the link.
- [ ] Submit both links (notebook + video) individually, ahead of the deadline.

## 13. References

1. Rossler, A., Cozzolino, D., Verdoliva, L., Riess, C., Thies, J., & Niessner, M. (2019). FaceForensics++:
   Learning to Detect Manipulated Facial Images. *ICCV*.
2. Li, Y., Yang, X., Sun, P., Qi, H., & Lyu, S. (2020). Celeb-DF: A Large-Scale Challenging Dataset for
   DeepFake Forensics. *CVPR*.
3. He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep Residual Learning for Image Recognition. *CVPR*. (ResNet)
4. Tan, M., & Le, Q. (2019). EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks. *ICML*.
5. Hochreiter, S., & Schmidhuber, J. (1997). Long Short-Term Memory. *Neural Computation*.
6. Güera, D., & Delp, E. J. (2018). Deepfake Video Detection Using Recurrent Neural Networks. *AVSS*.
7. Dolhansky, B., Bitton, J., Pflaum, B., Lu, J., Howes, R., Wang, M., & Ferrer, C. C. (2020). The DeepFake
   Detection Challenge (DFDC) Dataset. *arXiv:2006.07397*. (Dataset used in this project)
8. Dufour, N., & Gully, A. (2019). Contributing Data to Deepfake Detection Research. *Google AI Blog*.
   (DeepFake Detection (DFD) dataset, used in this project)
9. Yu, W., et al. (2023). YuNet: A Tiny Millisecond-level Face Detector. *Machine Intelligence Research*.
   (Face detector used for cropping)